# Notebook 3: Feature squeezing warm-start + DiffPure — AutoAttack + BPDA-EOT

## Evaluation protocol

**Identical attack setup to Notebook 1** — any difference in robust accuracy
is attributable solely to feature squeezing (color bit-depth + spatial) preprocessing.

**AutoAttack robust accuracy = min(standard, rand)**
- `standard`: APGD-CE, APGD-T, FAB-T, Square
  - APGD/FAB gradients are partially blind at the squeeze step (BPDA straight-through)
  - Square Attack (black-box) is unaffected and remains effective
- `rand`: APGD-CE, APGD-DLR with EOT (eot_iter=20)
  - Also partially blind at the squeeze step, but EOT helps with DiffPure stochasticity
- Taking min gives the strongest AutoAttack result

**BPDA-EOT** reported separately:
- Handles feature squeeze via BPDA straight-through on color quantization and spatial resize
- Handles DiffPure (stochastic) via EOT logit averaging
- This is the correct strong attack for the full squeeze+DiffPure pipeline

**Pipeline:** `x_adv → FeatureSqueeze(BPDA) → DiffPure → Standard WRN-28-10 → logits`

**t values:** t/T = 0.05, 0.075, 0.10 → t = 50, 75, 100


## 0. Install and clone

In [ ]:
# if %pip doesn't work, try replacing with !pip
%pip install robustbench torchsde einops --quiet
%pip install git+https://github.com/fra31/auto-attack
%pip install -U pip wheel ninja
!python -m pip install --no-cache-dir "setuptools<81"

import os
import json

# Remove the torch_extensions cache to force recompilation of CUDA extensions
# if os.path.exists('/root/.cache/torch_extensions'):
#     !rm -rf /root/.cache/torch_extensions
#     print("Cleared /root/.cache/torch_extensions to force re-compilation.")

if not os.path.exists('DiffPure'):
    !git clone https://github.com/NVlabs/DiffPure.git

import sys
sys.path.insert(0, 'DiffPure')
sys.path.insert(0, 'DiffPure/score_sde')

  Cloning https://github.com/fra31/auto-attack to /tmp/pip-req-build-lot45y70
  Running command git clone --filter=blob:none --quiet https://github.com/fra31/auto-attack /tmp/pip-req-build-lot45y70
  Resolved https://github.com/fra31/auto-attack to commit a39220048b3c9f2cca9a4d3a54604793c68eca7e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Mount Drive

In [ ]:
#these are specific to Colab — ignore if running locally or on another platform
from google.colab import drive
drive.mount('/content/drive') #

# UPDATE THIS PATH TO YOUR CHECKPOINT
CKPT_PATH = '/content/drive/MyDrive/checkpoint_8.pth'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. CONFIG

In [ ]:
import torch, random, numpy as np, os

# EXPERIMENT SETTINGS
# t/T = 0.05, 0.075, 0.10  →  t = 50, 75, 100
T_VALUES              = [50, 75, 100]

# Feature squeezing (Xu et al.): color bit-depth + spatial down/up before DiffPure
SQUEEZE_BITS          = 5       # per-channel quantization; use 8 for (near) identity
SQUEEZE_SPATIAL_SCALE = 0.5     # H,W -> floor(scale*H,W) -> back (CIFAR: 32->16->32)
SQUEEZE_ORDER         = "color_first"   # "color_first" or "spatial_first"

N_SAMPLES             = 64
ADV_BATCH_SIZE        = 16
ADV_EPS_LINF          = 8.0 / 255
ADV_EPS_L2            = 0.5
SEED                  = 42

# AutoAttack — custom version (APGD-CE from rand + Square from standard)
AA_APGD_N_ITER    = 10
AA_APGD_EOT_ITER  = 3
AA_APGD_N_RESTARTS = 1
AA_SQUARE_N_QUERIES = 500

BPDA_EOT_ATTACK_REPS  = 4    # EOT draws for gradient
BPDA_EOT_DEFENSE_REPS = 5   # EOT draws for verification

SCORE_TYPE            = 'score_sde'
SAMPLE_STEP           = 1
USE_BM                = False
LOG_DIR               = '/content/squeeze_diffpure_logs'


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

def set_seed(seed=SEED):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed()
os.makedirs(LOG_DIR, exist_ok=True)



Device: cuda


## 3. Load data

In [ ]:
from robustbench.data import load_cifar10

x_test, y_test = load_cifar10(n_examples=N_SAMPLES, data_dir='./data')
x_test, y_test = x_test.to(DEVICE), y_test.to(DEVICE)
print(f'x_test: {x_test.shape}, range [{x_test.min():.3f}, {x_test.max():.3f}]')

x_test: torch.Size([64, 3, 32, 32]), range [0.000, 1.000]


## 4. Load score_sde model

In [ ]:
import argparse
from score_sde.losses import get_optimizer
from score_sde.models import utils as mutils
from score_sde.models.ema import ExponentialMovingAverage
from score_sde import sde_lib
import score_sde.models.ncsnpp


def build_config(device):
    cfg = argparse.Namespace()
    cfg.data = argparse.Namespace()
    cfg.data.dataset = 'CIFAR10'; cfg.data.image_size = 32
    cfg.data.num_channels = 3;   cfg.data.centered = True

    cfg.model = argparse.Namespace()
    cfg.model.name = 'ncsnpp';              cfg.model.scale_by_sigma = False
    cfg.model.ema_rate = 0.9999;            cfg.model.normalization = 'GroupNorm'
    cfg.model.nonlinearity = 'swish';       cfg.model.nf = 128
    cfg.model.ch_mult = (1, 2, 2, 2);      cfg.model.num_res_blocks = 8
    cfg.model.attn_resolutions = (16,);     cfg.model.resamp_with_conv = True
    cfg.model.conditional = True;           cfg.model.fir = False
    cfg.model.fir_kernel = [1, 3, 3, 1];   cfg.model.skip_rescale = True
    cfg.model.resblock_type = 'biggan';     cfg.model.progressive = 'none'
    cfg.model.progressive_input = 'none';   cfg.model.progressive_combine = 'sum'
    cfg.model.attention_type = 'ddpm';      cfg.model.init_scale = 0.0
    cfg.model.fourier_scale = 16;           cfg.model.conv_size = 3
    cfg.model.sigma_min = 0.01;             cfg.model.sigma_max = 50
    cfg.model.num_scales = 1000;            cfg.model.beta_min = 0.1
    cfg.model.beta_max = 20.0;              cfg.model.dropout = 0.1
    cfg.model.embedding_type = 'positional'

    cfg.optim = argparse.Namespace()
    cfg.optim.optimizer = 'Adam'; cfg.optim.lr = 2e-4; cfg.optim.beta1 = 0.9
    cfg.optim.eps = 1e-8; cfg.optim.weight_decay = 0.0
    cfg.optim.warmup = 5000; cfg.optim.grad_clip = 1.0

    cfg.device = device; cfg.seed = SEED
    return cfg


def restore_checkpoint(ckpt_path, state, device):
    loaded = torch.load(ckpt_path, map_location=device)
    state['optimizer'].load_state_dict(loaded['optimizer'])
    state['model'].load_state_dict(loaded['model'], strict=False)
    state['ema'].load_state_dict(loaded['ema'])
    state['step'] = loaded['step']


def load_score_sde_model(ckpt_path, device):
    config    = build_config(device)
    model     = mutils.create_model(config)
    optimizer = get_optimizer(config, model.parameters())
    ema       = ExponentialMovingAverage(model.parameters(), decay=config.model.ema_rate)
    state     = dict(step=0, optimizer=optimizer, model=model, ema=ema)
    restore_checkpoint(ckpt_path, state, device)
    ema.copy_to(model.parameters())
    model.eval().to(device)
    print(f'Loaded score_sde model (step={state["step"]})')
    return model


score_model = load_score_sde_model(CKPT_PATH, DEVICE)

Loaded score_sde model (step=400005)


## 5. RevVPSDE and Standard classifier

In [ ]:
import torchsde
import torch.nn as nn
from robustbench.utils import load_model as rb_load_model


class RevVPSDE(nn.Module):
    def __init__(self, model, score_type='score_sde',
                 beta_min=0.1, beta_max=20.0, N=1000, img_shape=(3, 32, 32)):
        super().__init__()
        self.model = model; self.score_type = score_type
        self.img_shape = img_shape
        self.beta_0 = beta_min; self.beta_1 = beta_max; self.N = N
        self.discrete_betas = torch.linspace(beta_min / N, beta_max / N, N)
        self.alphas         = 1. - self.discrete_betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.vpsde          = sde_lib.VPSDE(beta_min=beta_min, beta_max=beta_max, N=N)
        self.noise_type = 'diagonal'
        self.sde_type   = 'ito'

    def vpsde_fn(self, t, x):
        beta_t = self.beta_0 + t * (self.beta_1 - self.beta_0)
        return -0.5 * beta_t[:, None] * x, torch.sqrt(beta_t)

    def rvpsde_fn(self, t, x, return_type='drift'):
        drift, diffusion = self.vpsde_fn(t, x)
        if return_type == 'drift':
            x_img    = x.view(-1, *self.img_shape)
            score_fn = mutils.get_score_fn(self.vpsde, self.model,
                                           train=False, continuous=True)
            score    = score_fn(x_img, t).view(x.shape[0], -1)
            return drift - diffusion[:, None] ** 2 * score
        return diffusion

    def f(self, t, x):
        return -self.rvpsde_fn(1 - t.expand(x.shape[0]), x, 'drift')

    def g(self, t, x):
        d = self.rvpsde_fn(1 - t.expand(x.shape[0]), x, 'diffusion')
        return d[:, None].expand(x.shape)


rev_vpsde      = RevVPSDE(score_model, SCORE_TYPE).to(DEVICE)
alphas_cumprod = rev_vpsde.alphas_cumprod.to(DEVICE)

classifier_inf = rb_load_model(
    model_name='Standard', dataset='cifar10', threat_model='Linf'
).eval().to(DEVICE)

classifier_l2 = rb_load_model(
    model_name='Standard', dataset='cifar10', threat_model='L2'
).eval().to(DEVICE)

classifier_dict = {
    'l_inf': [classifier_inf, 'Linf', ADV_EPS_LINF],
    'l_2': [classifier_l2, 'L2', ADV_EPS_L2],
}

print('RevVPSDE and Standard WRN-28-10 loaded.')

RevVPSDE and Standard WRN-28-10 loaded.


## 6. Feature squeezing with BPDA straight-through

- **Color squeeze**: quantize each channel in `[0,1]` to `2^b` levels (forward uses `round`; backward is identity — BPDA).
- **Spatial squeeze**: `area` downsample then `bilinear` upsample to original size; backward straight-through (BPDA).
- **Order**: `SQUEEZE_ORDER` is either `color_first` or `spatial_first`.

Used by BPDA-EOT and AutoAttack gradient-based attacks so gradients do not vanish at the squeeze steps. Square Attack ignores BPDA (black-box).


In [ ]:
import torch.nn.functional as F


def _color_squeeze_forward(x, bits):
    bits = int(bits)
    if bits >= 8:
        return x
    levels = float(2 ** bits - 1)
    return torch.round(x.clamp(0, 1) * levels) / levels


def _spatial_squeeze_forward(x, scale):
    _, _, H, W = x.shape
    h = max(1, int(H * scale))
    w = max(1, int(W * scale))
    x_small = F.interpolate(x, size=(h, w), mode='area')
    return F.interpolate(x_small, size=(H, W), mode='bilinear', align_corners=False)


class ColorSqueezeFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bits):
        return _color_squeeze_forward(x, int(bits))

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None


class SpatialSqueezeFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, scale):
        return _spatial_squeeze_forward(x, float(scale))

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None


def color_squeeze_bpda(x, bits):
    return ColorSqueezeFunction.apply(x, int(bits))


def spatial_squeeze_bpda(x, scale):
    if float(scale) >= 1.0 - 1e-6:
        return x
    return SpatialSqueezeFunction.apply(x, float(scale))


def feature_squeeze_bpda(x, bits, scale, order):
    if order not in ('color_first', 'spatial_first'):
        raise ValueError('SQUEEZE_ORDER must be "color_first" or "spatial_first"')
    if order == 'color_first':
        x = color_squeeze_bpda(x, bits)
        x = spatial_squeeze_bpda(x, scale)
    else:
        x = spatial_squeeze_bpda(x, scale)
        x = color_squeeze_bpda(x, bits)
    return x


print('Feature squeeze BPDA defined.')



Feature squeeze BPDA defined.


## 7. DiffPure purification

In [ ]:
def run_diffpure(x_in, rev_vpsde, alphas_cumprod, t_steps, device,
                 sample_step=1, use_bm=False):
    """
    DiffPure reverse SDE. x_in in [0,1] → purified in [0,1].
    Called WITHOUT torch.no_grad() so AutoAttack APGD/FAB get
    real gradients through sdeint_adjoint.
    BPDA-EOT is unaffected — it detaches the output itself.
    """
    x0 = (x_in * 2.0 - 1.0).to(device)
    B, C, H, W = x0.shape
    for _ in range(sample_step):
        e       = torch.randn_like(x0)
        a       = alphas_cumprod[t_steps - 1]
        x_noisy = x0 * a.sqrt() + e * (1.0 - a).sqrt()
        t0      = 1.0 - t_steps / 1000.0
        t1      = 1.0 - 1e-5
        ts      = torch.tensor([t0, t1], device=device)
        x_      = x_noisy.view(B, -1)
        if use_bm:
            bm  = torchsde.BrownianInterval(t0=t0, t1=t1,
                      size=(B, C * H * W), device=device)
            xs_ = torchsde.sdeint_adjoint(rev_vpsde, x_, ts, method='euler', bm=bm)
        else:
            xs_ = torchsde.sdeint_adjoint(rev_vpsde, x_, ts, method='euler')
        x0 = xs_[-1].view(B, C, H, W)
    return ((x0 + 1.0) / 2.0).clamp(0.0, 1.0)


print('DiffPure purification defined.')

DiffPure purification defined.


## 8. Feature squeeze + DiffPure model wrapper

- `mode='purify'`   → FeatureSqueeze(BPDA) → DiffPure; called by `BPDA_EOT_Attack`
- `mode='classify'` → classifier only; called by `BPDA_EOT_Attack` on detached image
- `mode='forward'`  → full pipeline, no `torch.no_grad()`; called by AutoAttack


In [ ]:
class FeatureSqueezeDiffPureModel(nn.Module):
    def __init__(self, classifier, rev_vpsde, alphas_cumprod,
                 t_steps, squeeze_bits, spatial_scale, squeeze_order,
                 device, sample_step=1, use_bm=False):
        super().__init__()
        self.classifier     = classifier
        self.rev_vpsde      = rev_vpsde
        self.alphas_cumprod = alphas_cumprod
        self.t_steps        = t_steps
        self.squeeze_bits   = int(squeeze_bits)
        self.spatial_scale  = float(spatial_scale)
        self.squeeze_order  = squeeze_order
        self.device         = device
        self.sample_step    = sample_step
        self.use_bm         = use_bm

    def purify(self, x):
        x_sq = feature_squeeze_bpda(
            x, self.squeeze_bits, self.spatial_scale, self.squeeze_order)
        return run_diffpure(
            x_sq, self.rev_vpsde, self.alphas_cumprod,
            self.t_steps, self.device, self.sample_step, self.use_bm
        )

    def forward(self, x, mode='forward'):
        if mode == 'purify':
            return self.purify(x)
        elif mode == 'classify':
            return self.classifier(x)
        elif mode == 'forward':
            return self.classifier(self.purify(x))
        else:
            raise NotImplementedError(mode)


print('FeatureSqueezeDiffPureModel defined.')



FeatureSqueezeDiffPureModel defined.


## 9. BPDA_EOT_Attack (from DiffPure repo, unmodified)

In [ ]:
import torch.nn.functional as F

_criterion = torch.nn.CrossEntropyLoss()


class BPDA_EOT_Attack():
    def __init__(self, model, norm, adv_eps=8.0/255, eot_defense_reps=10, eot_attack_reps=8):
        self.model = model
        self.config = {
            'eot_defense_ave'  : 'logits',
            'eot_attack_ave'   : 'logits',
            'eot_defense_reps' : eot_defense_reps,
            'eot_attack_reps'  : eot_attack_reps,
            'adv_steps'        : 50,
            'adv_norm'         : norm,
            'adv_eps'          : adv_eps,
            'adv_eta'          : 2.0 / 255,
            'log_freq'         : 10,
        }
        print(f'BPDA_EOT config: {self.config}')

    def purify(self, x):
        return self.model(x, mode='purify')

    def eot_defense_prediction(self, logits, reps=1, eot_defense_ave=None):
        if eot_defense_ave == 'logits':
            logits_pred = logits.view([reps, int(logits.shape[0]/reps), logits.shape[1]]).mean(0)
        elif eot_defense_ave == 'softmax':
            logits_pred = F.softmax(logits, dim=1).view([reps, int(logits.shape[0]/reps), logits.shape[1]]).mean(0)
        elif eot_defense_ave == 'logsoftmax':
            logits_pred = F.log_softmax(logits, dim=1).view([reps, int(logits.shape[0]/reps), logits.shape[1]]).mean(0)
        elif reps == 1:
            logits_pred = logits
        else:
            raise RuntimeError('Invalid eot_defense_ave')
        _, y_pred = torch.max(logits_pred, 1)
        return y_pred

    def eot_attack_loss(self, logits, y, reps=1, eot_attack_ave='loss'):
        if eot_attack_ave == 'logits':
            logits_loss = logits.view([reps, int(logits.shape[0]/reps), logits.shape[1]]).mean(0)
            y_loss = y
        elif eot_attack_ave == 'softmax':
            logits_loss = torch.log(F.softmax(logits, dim=1).view([reps, int(logits.shape[0]/reps), logits.shape[1]]).mean(0))
            y_loss = y
        elif eot_attack_ave == 'logsoftmax':
            logits_loss = F.log_softmax(logits, dim=1).view([reps, int(logits.shape[0]/reps), logits.shape[1]]).mean(0)
            y_loss = y
        elif eot_attack_ave == 'loss':
            logits_loss = logits
            y_loss = y.repeat(reps)
        else:
            raise RuntimeError('Invalid eot_attack_ave')
        return _criterion(logits_loss, y_loss)

    def predict(self, X, y, requires_grad=True, reps=1, eot_defense_ave=None, eot_attack_ave='loss'):
        if requires_grad:
            logits = self.model(X, mode='classify')
        else:
            with torch.no_grad():
                logits = self.model(X.data, mode='classify')
        y_pred  = self.eot_defense_prediction(logits.detach(), reps, eot_defense_ave)
        correct = torch.eq(y_pred, y)
        loss    = self.eot_attack_loss(logits, y, reps, eot_attack_ave)
        return correct.detach(), loss

    def pgd_update(self, X_adv, grad, X, adv_norm, adv_eps, adv_eta, eps=1e-10):
        if adv_norm == 'l_inf':
            X_adv.data += adv_eta * torch.sign(grad)
            X_adv = torch.clamp(torch.min(X + adv_eps, torch.max(X - adv_eps, X_adv)), 0, 1)
        elif adv_norm == 'l_2':
            X_adv.data += adv_eta * grad / grad.view(X.shape[0], -1).norm(p=2, dim=1).view(X.shape[0], 1, 1, 1)
            dists = (X_adv - X).view(X.shape[0], -1).norm(dim=1, p=2).view(X.shape[0], 1, 1, 1)
            X_adv = torch.clamp(X + torch.min(dists, adv_eps * torch.ones_like(dists)) * (X_adv - X) / (dists + eps), 0, 1)
        else:
            raise RuntimeError('Invalid adv_norm')
        return X_adv

    def purify_and_predict(self, X, y, purify_reps=1, requires_grad=True):
        X_repeat          = X.repeat([purify_reps, 1, 1, 1])
        X_repeat_purified = self.purify(X_repeat).detach().clone()
        X_repeat_purified.requires_grad_()
        correct, loss = self.predict(X_repeat_purified, y, requires_grad, purify_reps,
                                     self.config['eot_defense_ave'], self.config['eot_attack_ave'])
        if requires_grad:
            X_grads     = torch.autograd.grad(loss, [X_repeat_purified])[0]
            attack_grad = X_grads.view([purify_reps] + list(X.shape)).mean(dim=0)
            return correct, attack_grad
        return correct, None

    def eot_defense_verification(self, X_adv, y, correct, defended):
        for i in range(correct.nelement()):
            if correct[i] == 0 and defended[i] == 1:
                defended[i] = self.purify_and_predict(
                    X_adv[i].unsqueeze(0), y[i].view([1]),
                    self.config['eot_defense_reps'], requires_grad=False
                )[0]
        return defended

    def eval_and_bpda_eot_grad(self, X_adv, y, defended, requires_grad=True):
        correct, attack_grad = self.purify_and_predict(
            X_adv, y, self.config['eot_attack_reps'], requires_grad
        )
        if self.config['eot_defense_reps'] > 0:
            defended = self.eot_defense_verification(X_adv, y, correct, defended)
        else:
            defended *= correct
        return defended, attack_grad

    def attack_batch(self, X, y):
        defended = self.eval_and_bpda_eot_grad(X, y, torch.ones_like(y).bool(), False)[0]
        print(f'Baseline: {defended.sum()} / {len(defended)}')
        class_batch    = torch.zeros([self.config['adv_steps'] + 2, X.shape[0]]).bool()
        class_batch[0] = defended.cpu()
        ims_adv_batch  = torch.zeros(X.shape)
        for i in range(defended.nelement()):
            if not defended[i]: ims_adv_batch[i] = X[i].cpu()
        X_adv = X.clone()
        for step in range(self.config['adv_steps'] + 1):
            defended, attack_grad = self.eval_and_bpda_eot_grad(X_adv, y, defended)
            class_batch[step + 1] = defended.cpu()
            for i in range(defended.nelement()):
                if class_batch[step, i] == 1 and defended[i] == 0:
                    ims_adv_batch[i] = X_adv[i].cpu()
            if step < self.config['adv_steps']:
                X_adv = self.pgd_update(X_adv, attack_grad, X,
                    self.config['adv_norm'], self.config['adv_eps'], self.config['adv_eta'])
                X_adv = X_adv.detach().clone()
            if step == 1 or step % self.config['log_freq'] == 0 or step == self.config['adv_steps']:
                print(f'Step {step}/{self.config["adv_steps"]}  defended: {int(defended.sum())} / {X.shape[0]}')
            if int(defended.sum()) == 0:
                print('All fooled — stopping early.'); break
        for i in range(defended.nelement()):
            if defended[i]: ims_adv_batch[i] = X_adv[i].cpu()
        return class_batch, ims_adv_batch

    def attack_all(self, X, y, batch_size):
        class_path = torch.zeros([self.config['adv_steps'] + 2, 0]).bool()
        ims_adv    = torch.zeros(0)
        n_batches  = max(1, X.shape[0] // batch_size)
        for i in range(n_batches):
            xb = X[i*batch_size:min((i+1)*batch_size, X.shape[0])].clone().to(X.device)
            yb = y[i*batch_size:min((i+1)*batch_size, X.shape[0])].clone().to(X.device)
            cb, ib     = self.attack_batch(xb.contiguous(), yb.contiguous())
            class_path = torch.cat((class_path, cb), dim=1)
            ims_adv    = torch.cat((ims_adv, ib), dim=0)
            print(f'Finished batch {i}')
        return class_path, ims_adv


print('BPDA_EOT_Attack defined.')

BPDA_EOT_Attack defined.


## 10. AutoAttack wrapper and helper

In [ ]:
from autoattack import AutoAttack


class AAWrapper(nn.Module):
    """Wraps FeatureSqueezeDiffPureModel for AutoAttack. Gradients flow via squeeze BPDA and sdeint_adjoint."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x, mode='forward')


def run_autoattack(model, x_test, y_test, norm, eps, batch_size, log_path):
    """
    Custom AutoAttack: APGD-CE (rand, EOT) + Square Attack.
    - APGD-CE with EOT is the principled gradient attack for stochastic defenses
    - Square is black-box so unaffected by squeeze BPDA / non-differentiability of the true squeeze
    - Reduced parameters for compute budget
    """
    adversary = AutoAttack(
        AAWrapper(model), norm=norm, eps=eps,
        version='custom',
        attacks_to_run=['apgd-ce', 'square'],
        device=DEVICE, log_path=log_path,
    )
    adversary.apgd.n_iter     = AA_APGD_N_ITER
    adversary.apgd.eot_iter   = AA_APGD_EOT_ITER
    adversary.apgd.n_restarts = AA_APGD_N_RESTARTS
    adversary.square.n_queries = AA_SQUARE_N_QUERIES

    x_adv = adversary.run_standard_evaluation(x_test, y_test, bs=batch_size)

    correct = 0
    with torch.no_grad():
        for i in range(0, len(x_test), batch_size):
            xb = x_adv[i:i + batch_size].to(DEVICE)
            yb = y_test[i:i + batch_size].to(DEVICE)
            correct += (model(xb, mode='forward').argmax(1) == yb).sum().item()
    return correct / len(x_test)

print('AutoAttack wrapper defined.')



AutoAttack wrapper defined.


## 11. Baseline: clean accuracy

In [ ]:
set_seed(SEED)
correct1 = 0
correct2 = 0
with torch.no_grad():
    for i in range(0, N_SAMPLES, ADV_BATCH_SIZE):
        xb = x_test[i:i + ADV_BATCH_SIZE]
        yb = y_test[i:i + ADV_BATCH_SIZE]
        correct1 += (classifier_inf(xb).argmax(1) == yb).sum().item()
clean_acc_lf = correct1 / N_SAMPLES
print(f'Clean accuracy Linf(no attack, no defense): {clean_acc_lf:.2%}')


set_seed(SEED)
with torch.no_grad():
    for i in range(0, N_SAMPLES, ADV_BATCH_SIZE):
        xb = x_test[i:i + ADV_BATCH_SIZE]
        yb = y_test[i:i + ADV_BATCH_SIZE]
        correct2 += (classifier_l2(xb).argmax(1) == yb).sum().item()
clean_acc_l2 = correct2 / N_SAMPLES
print(f'Clean accuracy L2(no attack, no defense): {clean_acc_l2:.2%}')

Clean accuracy Linf(no attack, no defense): 92.19%
Clean accuracy L2(no attack, no defense): 92.19%


In [ ]:
linf_res = {}
l2_res ={}

## 12. Run all attacks at each t

In [48]:
import time

SQUEEZE_SCALE_TAG = str(SQUEEZE_SPATIAL_SCALE).replace('.', 'p')

results = {}

for k, (classifier, norm, ADV_EPS) in classifier_dict.items():
    for t_val in T_VALUES:
        print(f'\n{"="*60}')
        print(f'  t = {t_val}  (t/T = {t_val/1000:.3f})  squeeze bits={SQUEEZE_BITS} scale={SQUEEZE_SPATIAL_SCALE} order={SQUEEZE_ORDER}')
        print(f'{"="*60}')

        set_seed(SEED)

        model_t = FeatureSqueezeDiffPureModel(
            classifier=classifier, rev_vpsde=rev_vpsde,
            alphas_cumprod=alphas_cumprod, t_steps=t_val,
            squeeze_bits=SQUEEZE_BITS, spatial_scale=SQUEEZE_SPATIAL_SCALE,
            squeeze_order=SQUEEZE_ORDER, device=DEVICE,
            sample_step=SAMPLE_STEP, use_bm=USE_BM,
        ).eval().to(DEVICE)

        # ── Clean accuracy under defense ──────────────────────────────────
        correct_clean = 0
        with torch.no_grad():
            for i in range(0, N_SAMPLES, ADV_BATCH_SIZE):
                xb = x_test[i:i + ADV_BATCH_SIZE]
                yb = y_test[i:i + ADV_BATCH_SIZE]
                correct_clean += (model_t(xb, mode='forward').argmax(1) == yb).sum().item()
        acc_defense = correct_clean / N_SAMPLES
        print(f'  Clean acc under DiffPure - {norm} - {ADV_EPS}: {acc_defense:.2%}')

        # ══ AutoAttack (APGD-CE + Square) ════════════════════════════════
        print(f'\n  [AutoAttack - {norm} - {ADV_EPS}] APGD-CE (n_iter={AA_APGD_N_ITER}, eot={AA_APGD_EOT_ITER}) + Square (n_queries={AA_SQUARE_N_QUERIES})')
        set_seed(SEED); t0 = time.time()
        robust_aa = run_autoattack(
            model_t, x_test, y_test, norm, ADV_EPS,
            batch_size=ADV_BATCH_SIZE,
            log_path=f'{LOG_DIR}/aa_std_sq{SQUEEZE_BITS}_s{SQUEEZE_SCALE_TAG}_t{t_val}.txt',
        )
        at_final_time = time.time() - t0
        print(f'  AutoAttack - {norm} - {ADV_EPS}] robust acc: {robust_aa:.2%}  ({at_final_time:.0f}s)')

        # ══ BPDA-EOT (reported separately) ═══════════════════════════════
        print(f'\n  [BPDA-EOT] PGD 50 steps, EOT={BPDA_EOT_ATTACK_REPS} draws')
        print(f'  Note: BPDA handles feature squeeze + DiffPure (stochastic)')
        set_seed(SEED); t0 = time.time()
        adversary_bpda = BPDA_EOT_Attack(
            model_t, norm=k, adv_eps=ADV_EPS,
            eot_attack_reps=BPDA_EOT_ATTACK_REPS,
            eot_defense_reps=BPDA_EOT_DEFENSE_REPS,
        )
        class_path, _ = adversary_bpda.attack_all(x_test, y_test, batch_size=ADV_BATCH_SIZE)
        robust_bpda   = float(class_path[-1].sum()) / N_SAMPLES
        bpda_final_time = time.time() - t0
        print(f'  [BPDA-EOT - {norm} - {ADV_EPS}]    robust acc: {robust_bpda:.2%}  ({bpda_final_time:.0f}s)')

        if k == 'l_inf':
            linf_res[t_val] = {
                'clean_no_defense'  : clean_acc_lf,
                'clean_with_defense': acc_defense,
                'robust_aa'         : robust_aa,
                'robust_bpda_eot'   : robust_bpda,
                'aa_time_sec'       : at_final_time,
                'bpda_time_sec'     : bpda_final_time,
            }
            with open(f'{LOG_DIR}/results_t{t_val}_{k}.json', 'w') as f:
              json.dump(linf_res, f)
        else:
            l2_res[t_val] = {
                'clean_no_defense'  : clean_acc_l2,
                'clean_with_defense': acc_defense,
                'robust_aa'         : robust_aa,
                'robust_bpda_eot'   : robust_bpda,
                'aa_time_sec'       : at_final_time,
                'bpda_time_sec'     : bpda_final_time,
            }
            with open(f'{LOG_DIR}/results_t{t_val}_{k}.json', 'w') as f:
              json.dump(l2_res, f)




  t = 50  (t/T = 0.050)  squeeze bits=5 scale=0.5 order=color_first
  Clean acc under DiffPure - Linf - 0.03137254901960784: 70.31%

  [AutoAttack - Linf - 0.03137254901960784] APGD-CE (n_iter=10, eot=3) + Square (n_queries=500)
using custom version including apgd-ce, square.
initial accuracy: 73.44%
apgd-ce - 1/3 - 3 out of 16 successfully perturbed
apgd-ce - 2/3 - 3 out of 16 successfully perturbed
apgd-ce - 3/3 - 5 out of 15 successfully perturbed
robust accuracy after APGD-CE: 56.25% (total time 1639.6 s)
square - 1/3 - 1 out of 16 successfully perturbed
square - 2/3 - 1 out of 16 successfully perturbed
square - 3/3 - 0 out of 4 successfully perturbed
robust accuracy after SQUARE: 53.12% (total time 6289.2 s)
max Linf perturbation: 0.03137, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 53.12%
  AutoAttack - Linf - 0.03137254901960784] robust acc: 73.44%  (6335s)

  [BPDA-EOT] PGD 50 steps, EOT=4 draws
  Note: BPDA handles feature squeeze + DiffPure (stochastic)
BPD

## 13. Summary

In [49]:
print('\n' + '='*72)
print('  RESULTS — Feature squeeze + DiffPure (SDE) + Standard WRN-28-10')
print(f'  L∞ ε=8/255  bits={SQUEEZE_BITS} spatial={SQUEEZE_SPATIAL_SCALE} order={SQUEEZE_ORDER}  n={N_SAMPLES}  seed={SEED}')
print('='*72)
print(f'  Clean acc (no attack, no defense): {clean_acc_lf:.2%}')
print()
print(f'  {"t":>5}  {"t/T":>6}  {"clean":>8}  {"AA":>8}  {"BPDA-EOT":>10} {"AA time":>8}  {"BPDA time":>10}')
print(f'  {"-"*5}  {"-"*6}  {"-"*8}  {"-"*8}  {"-"*8}  {"-"*8}  {"-"*10} {"-"*8}  {"-"*10}')
for t_val, r in linf_res.items():
    print(f'  {t_val:>5}  {t_val/1000:>6.3f}  '
          f'{r["clean_with_defense"]:>8.2%}  '
          f'{r["robust_aa"]:>8.2%}  '
          f'{r["robust_bpda_eot"]:>10.2%}  '
          f'{r["aa_time_sec"]:>8.0f}  '
          f'{r["bpda_time_sec"]:>8.0f}')
print('='*72)




  RESULTS — Feature squeeze + DiffPure (SDE) + Standard WRN-28-10
  L∞ ε=8/255  bits=5 spatial=0.5 order=color_first  n=64  seed=42
  Clean acc (no attack, no defense): 92.19%

      t     t/T     clean        AA    BPDA-EOT  AA time   BPDA time
  -----  ------  --------  --------  --------  --------  ---------- --------  ----------
     50   0.050    70.31%    73.44%      48.44%      6335      1119
     75   0.075    75.00%    68.75%      64.06%      9428      1573
    100   0.100    75.00%    76.56%      62.50%     12619      2070


In [50]:
print('\n' + '='*72)
print('  RESULTS — Feature squeeze + DiffPure (SDE) + Standard WRN-28-10')
print(f'  L2 ε={ADV_EPS_L2}  bits={SQUEEZE_BITS} spatial={SQUEEZE_SPATIAL_SCALE} order={SQUEEZE_ORDER}  n={N_SAMPLES}  seed={SEED}')
print('='*72)
print(f'  Clean acc (no attack, no defense): {clean_acc_l2:.2%}')
print()
print(f'  {"t":>5}  {"t/T":>6}  {"clean":>8}  {"AA":>8}  {"BPDA-EOT":>10} {"AA time":>8}  {"BPDA time":>10}')
print(f'  {"-"*5}  {"-"*6}  {"-"*8}  {"-"*8}  {"-"*8}  {"-"*8}  {"-"*10} {"-"*8}  {"-"*10}')
for t_val, r in l2_res.items():
    print(f'  {t_val:>5}  {t_val/1000:>6.3f}  '
          f'{r["clean_with_defense"]:>8.2%}  '
          f'{r["robust_aa"]:>8.2%}  '
          f'{r["robust_bpda_eot"]:>10.2%}  '
          f'{r["aa_time_sec"]:>8.0f}  '
          f'{r["bpda_time_sec"]:>8.0f}')
print('='*72)




  RESULTS — Feature squeeze + DiffPure (SDE) + Standard WRN-28-10
  L2 ε=0.5  bits=5 spatial=0.5 order=color_first  n=64  seed=42
  Clean acc (no attack, no defense): 92.19%

      t     t/T     clean        AA    BPDA-EOT  AA time   BPDA time
  -----  ------  --------  --------  --------  --------  ---------- --------  ----------
     50   0.050    70.31%    68.75%      67.19%      5878       991
     75   0.075    75.00%    78.12%      70.31%      9625      1499
    100   0.100    75.00%    78.12%      65.62%     12777      1968


In [51]:
import shutil
import os

# Define the source directory to zip
source_dir = LOG_DIR # This is '/content/squeeze_diffpure_logs'

# Define the destination path in Google Drive
drive_path = os.path.dirname(CKPT_PATH) # This should be '/content/drive/MyDrive'
output_filename = os.path.join(drive_path, 'squeeze_diffpure_logs_archive')

# Create the zip archive
print(f"Zipping '{source_dir}' to '{output_filename}.zip'...")
shutil.make_archive(output_filename, 'zip', source_dir)

print(f"Successfully zipped '{source_dir}' to '{output_filename}.zip' and saved to Google Drive.")

Zipping '/content/squeeze_diffpure_logs' to '/content/drive/MyDrive/squeeze_diffpure_logs_archive.zip'...
Successfully zipped '/content/squeeze_diffpure_logs' to '/content/drive/MyDrive/squeeze_diffpure_logs_archive.zip' and saved to Google Drive.
